# 🚀 Google Colab GPU & Machine Learning 实验工作台 (AI Ultra 5x ~1,000 CU/月)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davidshizw/stock-colab/blob/main/notebooks/02_Colab_GPU_ML_Factor_Ranking_and_Regimes.ipynb)

本 Notebook 对应平台 **第 5 节（Machine Learning and GPU Research）**，严格遵循：
1. **从简单统计基线（Ridge L2）开始**，再引入非线性梯度提升树（GBDT / LightGBM）多因子交互建模。
2. **Purged & Embargoed 时序切分**：严格隔离训练集（In-Sample）、验证集（Validation）与样本外测试集（Out-of-Sample），防止重叠收益窗口泄露。
3. **算力预算监控与 Google Drive Checkpoint 持久化**：自动检测 Colab GPU（T4 / L4 / A100）并追踪每月 1,000 Compute Units 消耗。

In [1]:
# @title 1. 环境初始化（解压最新内嵌源码 + 安装依赖）
import base64, io, os, subprocess, sys, tarfile
from pathlib import Path

payload = "..."
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(payload)), mode="r:gz") as tf:
    tf.extractall(".")
print("✅ Synced latest quant_platform source package into Colab workspace")

if "google.colab" in sys.modules:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "duckdb>=1.0.0", "pyarrow>=16.0.0", "pydantic>=2.7.0", "pydantic-settings>=2.2.0",
        "scipy>=1.13.0", "scikit-learn>=1.5.0", "lightgbm>=4.3.0", "yfinance>=0.2.40",
        "fastapi>=0.111.0", "uvicorn>=0.30.0", "httpx>=0.27.0"
    ])
if str(Path("src").resolve()) not in sys.path:
    sys.path.insert(0, str(Path("src").resolve()))

from quant_platform.colab.runtime import ColabSessionManager
colab_mgr = ColabSessionManager()
print("Colab Runtime Status:", colab_mgr.get_compute_unit_status())


Hardware Profile: {'in_colab': False, 'device': 'cpu', 'accelerator_name': 'CPU', 'tier': 'CPU', 'vram_gb': 0.0, 'estimated_cu_per_hour': 0.08}
Compute Unit & Storage Status: {'accelerator': 'CPU', 'tier': 'CPU', 'in_colab': False, 'storage_root': 'data_store', 'cu_rate_per_hour': 0.08, 'session_elapsed_minutes': 0.0, 'session_cu_used': 0.0, 'monthly_cu_budget': 1000.0, 'total_cu_used': 0.0, 'remaining_cu': 1000.0}


In [2]:
# 2. 加载或初始化 DuckDB + Parquet 跨市场面板数据
store = MarketDataStore(db_path=colab_mgr.duckdb_path, parquet_dir=colab_mgr.parquet_dir)
end_date = date(2026, 11, 6)
start_date = end_date - timedelta(days=500)

if store.check_data_quality().total_price_rows == 0:
    build_default_pipeline(store, mode='fixture').run_ingestion(start_date=start_date, end_date=end_date, incremental=False)

runner = ColabMLExperimentRunner(store=store, colab_session=colab_mgr, seed=42)

In [3]:
# 3. 实验一：截面股票收益排序 (Ridge 统计基线 vs 非线性多因子交互 GBDT)
import json
ranking_exp = runner.run_stock_ranking_experiments(start_date=start_date, end_date=end_date, market='ALL')
print(json.dumps(ranking_exp, indent=2, ensure_ascii=False))

{
  "panel_rows": 630,
  "features_used": [
    "value_score",
    "quality_score",
    "momentum_score",
    "low_risk_score",
    "liquidity_score",
    "market_specific_score",
    "mom_1m_z",
    "mom_3m_z",
    "roe_z",
    "volatility_20d_z"
  ],
  "ridge_baseline": {
    "val_rank_ic": 0.097,
    "oos_rank_ic": 0.0642
  },
  "gbdt_interaction_model": {
    "val_rank_ic": -0.1244,
    "oos_rank_ic": -0.0525
  },
  "feature_importance": {
    "value_score": 0.008,
    "quality_score": 0.0823,
    "momentum_score": 0.1973,
    "low_risk_score": 0.0165,
    "liquidity_score": 0.044,
    "market_specific_score": 0.1164,
    "mom_1m_z": 0.344,
    "mom_3m_z": 0.0945,
    "roe_z": 0.0772,
    "volatility_20d_z": 0.0197
  },
  "compute_status": {
    "accelerator": "CPU",
    "tier": "CPU",
    "in_colab": false,
    "storage_root": "data_store",
    "cu_rate_per_hour": 0.08,
    "session_elapsed_minutes": 0.05,
    "session_cu_used": 0.0001,
    "monthly_cu_budget": 1000.0,
    "total_

In [4]:
# 4. 实验二：市场状态分类 (Market Regime Classification) 与 20 日前瞻波动率预测 (Volatility Forecasting)
vol_regime_exp = runner.run_volatility_and_regime_experiments(start_date=start_date, end_date=end_date, market='ALL')
print(json.dumps(vol_regime_exp, indent=2, ensure_ascii=False))

{
  "volatility_forecast_oos_mae": 0.0262,
  "volatility_forecast_oos_spearman": 0.696,
  "regime_classifier_oos_accuracy": 0.5132,
  "regime_distribution": {
    "BULL_LOW_VOL": 0.292,
    "BULL_HIGH_VOL": 0.275,
    "BEAR_HIGH_VOL": 0.225,
    "BEAR_LOW_VOL": 0.208
  }
}
